# GPT 모델 직접 구현 - 실습 코드 1: GPT 모델 구현

- Tutorial ID: `expand-gpt-from-scratch`
- Tutorial: GPT 모델 직접 구현
- Section ID: `expand-gpt-from-scratch-code-1`
- Section: 실습 코드 1: GPT 모델 구현

## 이 노트북에서 배우는 것

이 실습에서는 GPT(Generative Pre-trained Transformer)를 **PyTorch만 사용해서 밑바닥부터** 직접 구현합니다. `transformers` 같은 라이브러리로 완성된 모델을 불러오는 대신, 다음 두 가지를 코드로 직접 눈으로 확인하는 것이 목표입니다.

1. **Q(Query)/K(Key)/V(Value)가 어떤 shape으로 만들어지고, 그 shape이 어떻게 attention score로 이어지는지**
2. **미래 토큰을 `-inf`로 가린 뒤, softmax를 통과하면 그 확률이 정말로 0이 되는지**

이 두 가지를 직접 확인하고 나면 "GPT는 왜 다음 토큰을 예측할 때 미래를 훔쳐보지 않는지", "attention이 실제로 무엇을 계산하는지"를 코드 수준에서 설명할 수 있게 됩니다.

이 노트북은 처음 GPT 구조를 공부하는 분들을 위한 자료입니다. 새로운 개념이 나올 때마다 그 개념을 먼저 설명(마크다운 셀)하고, 바로 이어서 작은 숫자로 된 예제 코드로 직접 확인하는 순서로 구성했습니다. 코드는 반드시 위에서 아래로 순서대로 실행해주세요 (뒤에 나오는 셀이 앞에서 정의한 클래스와 변수를 그대로 사용합니다).

## 목차 (Table of Contents)

1. GPT 전체 구조 한눈에 보기
2. 하이퍼파라미터와 텐서 shape 이해하기
3. 토큰 임베딩(Token Embedding)과 위치 임베딩(Position Embedding)
4. Causal Mask(인과적 마스크) — 미래를 가리는 방법
5. Multi-Head Self-Attention 직접 구현하기
6. Feed-Forward Network(FFN)
7. LayerNorm과 잔차 연결(Residual Connection)
8. 전체 GPT 모델 조립하기
9. 텍스트 생성하기 (Autoregressive Generation)
10. (참고) PyTorch 내장 `nn.MultiheadAttention`과 비교
11. GPT-2 Small 규모로 파라미터 수 확인하기
12. 정리 및 다음 단계

## 코드 읽는 법

이 노트북은 "정답 코드를 한 번 실행"하고 끝나는 자료가 아니라, **수학/아키텍처 개념이 실제 텐서 연산으로 바뀌는 과정을 한 줄씩 추적**하기 위한 실습 노트입니다.

**읽는 순서 (권장)**

1. 각 섹션의 설명(마크다운)을 먼저 읽고, "이 코드가 왜 필요한가"를 이해합니다.
2. 코드 셀을 실행하면서 `print`로 출력되는 **shape과 값**을 직접 확인합니다. (직접 실행해보는 것이 이해에 훨씬 도움이 됩니다!)
3. 이해가 잘 안 가는 부분은 그 위의 "미니 예제" 셀로 돌아가서 작은 숫자로 다시 확인해봅니다.
4. 각 클래스 데모가 끝날 때마다 "이 클래스에 넣은 입력 shape과 나온 출력 shape이 같은가?"를 스스로 확인해봅니다. (같아야 여러 블록을 쌓을 수 있습니다.)

**작게 실험해보기**

이 노트북의 모든 데모는 아주 작은 숫자(`d_model=16`, `num_heads=2`, `seq_len=6` 등)로 실행되도록 만들어져 있습니다. 이렇게 하면 CPU에서도 빠르게 실행되고, 출력되는 텐서를 눈으로 직접 읽을 수 있습니다. 익숙해지면 아래와 같은 값들을 바꿔가며 실험해보세요.

- `num_heads`를 2 → 4로 바꾸면 각 헤드의 차원(`d_head`)은 어떻게 바뀔까요?
- `seq_len`을 늘리면 causal mask 행렬은 어떤 모양이 될까요?
- `temperature`를 0.1 / 1.0 / 2.0으로 바꾸면 생성되는 토큰의 분포가 어떻게 달라질까요?

**주의**

- 이 노트북은 GPU 없이 CPU에서도 실행되도록 작은 크기로 설계했습니다. 다만 마지막 섹션(GPT-2 Small, 약 1.24억 파라미터)은 파라미터 개수만 계산하며, 실제로 학습을 시키지는 않습니다.
- 이 노트북은 **모델 구조(architecture)** 에 집중합니다. 토크나이저, 학습 데이터, 학습 루프(`loss.backward()`, optimizer 등)는 다루지 않습니다 — 이 부분은 이후 실습에서 다룰 내용입니다.
- 아래 코드는 `torch`가 설치되어 있어야 실행됩니다 (Colab에는 기본 설치되어 있습니다). 로컬 환경이라면 `pip install torch`로 설치해주세요.

In [1]:
# 이 노트북에서 사용할 라이브러리를 불러옵니다.

import torch                      # 텐서 연산의 기본이 되는 라이브러리
import torch.nn as nn             # nn.Linear, nn.LayerNorm, nn.Embedding 등 신경망 구성 요소
import torch.nn.functional as F   # softmax, GELU 등 함수 형태의 연산
import math                       # attention score를 스케일링할 때 sqrt() 계산에 사용

# 실행할 때마다 같은 난수가 나오도록 seed를 고정합니다.
# (그래야 "제 결과와 노트북 결과가 달라요" 같은 혼란 없이,
#  값을 바꿔가며 실험할 때 재현 가능한 비교를 할 수 있습니다.)
torch.manual_seed(42)

print("torch version:", torch.__version__)

torch version: 2.12.1+cu130


## 1. GPT 전체 구조 한눈에 보기

세부 구현에 들어가기 전에, GPT가 입력부터 출력까지 어떤 순서로 데이터를 처리하는지 큰 그림부터 보고 시작하겠습니다. 아래 각 단계는 이 노트북에서 순서대로 하나씩 구현하고 확인해볼 것입니다.

```
입력 문장이 토크나이저를 거쳐 숫자로 바뀐 것: token_ids  [B, T]
        │
        ▼
  토큰 임베딩 + 위치 임베딩   ->  x  [B, T, d_model]
        │
        ▼
  ┌───────────────────────────────┐
  │   GPTBlock  (1번째)             │
  ├───────────────────────────────┤
  │   GPTBlock  (2번째)             │   <- Self-Attention + FFN으로 구성된 블록을
  ├───────────────────────────────┤      num_layers번 반복해서 쌓음
  │   ...                          │
  ├───────────────────────────────┤
  │   GPTBlock  (num_layers번째)    │
  └───────────────────────────────┘
        │
        ▼
   최종 LayerNorm (ln_f)
        │
        ▼
   Linear head: d_model -> vocab_size
        │
        ▼
   logits  [B, T, vocab_size]   <- "다음 토큰"에 대한 점수
```

핵심 아이디어는 단순합니다: **"지금까지 본 토큰들만 가지고, 각 위치에서 다음 토큰이 무엇일지 점수를 매긴다."** 이 노트북 전체는 이 한 문장을 실제 텐서 연산으로 어떻게 구현하는지를 다룹니다. 위 그림에 나온 각 화살표와 박스를 이 노트북의 섹션 2~8에서 하나씩 직접 만들어볼 것입니다.

## 2. 하이퍼파라미터와 텐서 shape 이해하기

GPT 코드를 처음 보면 `B`, `T`, `d_model` 같은 약자들이 정신없이 등장합니다. 본격적으로 모델을 구현하기 전에, 이 노트북 전체에서 반복해서 등장할 기호들을 먼저 정리하고 갑니다.

| 기호 | 의미 | 이 노트북의 예시 값 |
|---|---|---|
| `B` (batch_size) | 한 번에 모델에 넣는 "문장(시퀀스)"의 개수 | 2 |
| `T` (seq_len) | 시퀀스 하나의 토큰 개수 (문장 길이) | 6 |
| `vocab_size` | 모델이 알고 있는 전체 토큰(단어 조각)의 개수 | 50 (실제 GPT-2는 50,257) |
| `d_model` | 토큰 하나를 표현하는 벡터의 차원 수 (임베딩 차원) | 16 (실제 GPT-2 Small은 768) |
| `num_heads` | Self-Attention을 병렬로 몇 개의 "시선"으로 나눠 볼지 | 2 (실제 GPT-2 Small은 12) |
| `d_head` | 헤드 하나가 담당하는 차원 수 (`= d_model / num_heads`) | 8 |
| `d_ff` | Feed-Forward Network 내부의 확장된 차원 수 | 64 (실제 GPT-2 Small은 3072) |
| `num_layers` | GPTBlock을 몇 겹 쌓을지 | 2 (실제 GPT-2 Small은 12) |

이 노트북에서는 위 표의 "이 노트북의 예시 값"처럼 **아주 작은 숫자**를 사용합니다. 실제 GPT-2 Small 크기를 그대로 쓰면 print 결과가 너무 커서 눈으로 shape을 추적하기 어렵기 때문입니다. 노트북 맨 마지막 섹션(11번)에서만 실제 GPT-2 Small 크기로 모델을 만들어 파라미터 개수를 확인합니다.

아래 코드에서 `token_ids`는 실제 문장이 토크나이저를 거쳐 숫자로 바뀐 결과라고 가정한 것입니다. (토크나이저 자체는 이 노트북의 범위를 벗어나므로, 여기서는 `torch.randint`로 "이미 토큰화된 문장"을 흉내만 냅니다.)

In [2]:
# ---- 하이퍼파라미터 정의 -----------------------------------------
vocab_size = 50      # 예시용 작은 단어장 크기 (실제 GPT-2는 50257)
d_model = 16          # 임베딩 차원 (실제 GPT-2 Small은 768)
num_heads = 2         # attention head 개수 (실제 GPT-2 Small은 12)
d_ff = 64             # FFN 내부 차원 (실제 GPT-2 Small은 3072)
seq_len = 6            # 문장 길이 (토큰 개수)
batch_size = 2         # 한 번에 처리할 문장 개수
max_seq_len = 32      # 모델이 다룰 수 있는 최대 문장 길이 (position embedding 크기)

# ---- 가짜 입력 데이터 만들기 --------------------------------------
# 실제로는 "나는 오늘 학교에 간다" 같은 문장을 토크나이저가
# [3, 17, 42, 8, 91, 6] 같은 정수 배열로 바꿔줍니다.
# 여기서는 토크나이저 없이, 0~vocab_size 범위의 정수를 무작위로 뽑아
# "이미 토큰화된 문장 2개(batch_size=2)"라고 가정합니다.
token_ids = torch.randint(0, vocab_size, (batch_size, seq_len))

print("token_ids shape:", token_ids.shape)  # [B, T] = [2, 6]
print(token_ids)

token_ids shape: torch.Size([2, 6])
tensor([[42, 17, 26, 14, 26, 35],
        [20, 24,  0, 13, 28, 14]])


## 3. 토큰 임베딩(Token Embedding)과 위치 임베딩(Position Embedding)

`token_ids`에 들어있는 숫자(예: 17번 토큰)는 그 자체로는 아무 의미가 없는 "인덱스"일 뿐입니다. 모델이 다루기 좋은 형태로 바꾸려면, 이 인덱스를 **의미를 담은 벡터**로 바꿔줘야 합니다. 이 역할을 하는 것이 `nn.Embedding`입니다.

`nn.Embedding(vocab_size, d_model)`은 사실 **`(vocab_size, d_model)` 크기의 커다란 표(lookup table)** 일 뿐입니다. 예를 들어 `vocab_size=50`, `d_model=16`이면, 50개의 행(각 토큰마다 하나씩)과 16개의 열(각 토큰을 표현하는 숫자 16개)을 가진 표입니다. 토큰 번호 17이 들어오면, 그 표에서 **17번째 행을 그대로 꺼내오는 것**이 임베딩입니다. (이 표의 값들은 학습을 통해 점점 의미 있는 값으로 바뀌어 갑니다. 지금은 무작위 초기값입니다.)

그런데 Self-Attention은 구조상 "이 토큰이 몇 번째 위치에 있었는지"를 전혀 모릅니다. (뒤에서 자세히 보겠지만, attention은 모든 위치를 그저 집합처럼 다루기 때문입니다.) "나는 밥을 먹었다"와 "밥을 나는 먹었다"를 구별하려면 순서 정보가 필요하죠. 그래서 GPT는 토큰 임베딩에 **위치 임베딩(position embedding)** 을 더해서, "이 벡터는 3번째 위치에 있는 토큰이다"라는 정보를 함께 넣어줍니다.

- `tok_emb`: 토큰이 "무엇"인지에 대한 정보
- `pos_emb`: 토큰이 "어디"에 있는지에 대한 정보
- 최종 입력 `x = tok_emb(token_ids) + pos_emb(위치)`: "무엇이 어디에 있는지"를 합친 정보

GPT-2는 위치 정보를 수식으로 계산하는 대신(예: Transformer 원 논문의 sin/cos 방식), **위치 자체도 학습되는 임베딩**으로 둡니다 (`nn.Embedding(max_seq_len, d_model)`). 그래서 `max_seq_len`(모델이 다룰 수 있는 최대 길이)을 미리 정해둬야 합니다.

In [3]:
# nn.Embedding은 (vocab_size, d_model) 크기의 표를 만듭니다.
tok_emb = nn.Embedding(vocab_size, d_model)     # "토큰 -> 벡터" 표
pos_emb = nn.Embedding(max_seq_len, d_model)    # "위치 -> 벡터" 표

# 1) 토큰 임베딩: token_ids의 각 숫자를 d_model차원 벡터로 바꿉니다.
tok_e = tok_emb(token_ids)          # [B, T] -> [B, T, d_model]
print("tok_e shape:", tok_e.shape)  # [2, 6, 16]

# 2) 위치 임베딩: [0, 1, 2, ..., T-1] 위치마다 벡터 하나씩.
pos_ids = torch.arange(seq_len)     # tensor([0, 1, 2, 3, 4, 5])
pos_e = pos_emb(pos_ids)            # [T] -> [T, d_model]
print("pos_ids:", pos_ids)
print("pos_e shape:", pos_e.shape)  # [6, 16]  (모든 배치가 같은 위치 벡터를 공유합니다)

# 3) 두 임베딩을 더합니다.
# pos_e의 shape은 [T, d_model]이지만, tok_e는 [B, T, d_model]입니다.
# PyTorch의 broadcasting이 자동으로 pos_e를 배치 차원(B)만큼 복사해서 더해줍니다.
x = tok_e + pos_e
print("x shape:", x.shape)  # [2, 6, 16] -- 이제부터 이 x가 GPTBlock들을 통과하게 됩니다.

tok_e shape: torch.Size([2, 6, 16])
pos_ids: tensor([0, 1, 2, 3, 4, 5])
pos_e shape: torch.Size([6, 16])
x shape: torch.Size([2, 6, 16])


## 4. Causal Mask(인과적 마스크) — 미래를 가리는 방법

GPT는 **"지금까지 나온 토큰들을 보고, 다음 토큰을 예측"** 하도록 학습됩니다. 그런데 학습을 효율적으로 하기 위해, 문장 전체 `[T0, T1, T2, T3, T4]`를 한 번에 모델에 넣고, 각 위치에서 동시에 "다음 토큰이 뭘까?"를 예측하게 만듭니다.

여기서 문제가 생깁니다. Self-Attention은 기본적으로 모든 위치가 서로를 다 볼 수 있습니다. 만약 아무 제약이 없다면, 2번째 위치(`T1`)에서 다음 토큰을 예측할 때 미래인 `T2`, `T3`, `T4`까지 몰래 참고할 수 있게 됩니다 — 이건 시험 문제를 풀면서 답지를 미리 보는 것과 같습니다. 실제로 문장을 생성할 때는 미래 토큰이 아직 존재하지 않으므로, 학습 때도 미래를 볼 수 없게 **강제로 가려야** 합니다. 이것이 **causal mask(인과적 마스크)** 입니다.

**작동 원리는 아주 단순합니다.**

1. attention score(각 위치가 서로를 얼마나 볼지 나타내는 점수) 행렬에서, "미래 위치"에 해당하는 칸에 아주 작은 값(`-inf`, 음의 무한대)을 채워 넣습니다.
2. 그 다음 softmax를 적용하면, `-inf`는 `e^(-inf) = 0`이 되어 **확률이 정확히 0** 이 됩니다.
3. 결과적으로 각 위치는 "자기 자신과 그 이전 위치들"에만 확률을 나눠 갖게 되고, 미래 위치로는 정보가 전혀 흘러가지 않습니다.

아래 코드에서는 아직 실제 attention을 만들기 전에, **이 마스킹 아이디어만 따로 떼어내어** 아주 작은 5×5 예제로 확인해봅니다. (실제 attention score 대신, 설명을 위해 무작위 숫자를 "가짜 attention score"라고 가정해서 사용합니다.)

In [4]:
T_demo = 5  # 이해를 돕기 위해 아주 짧은 5개짜리 시퀀스로 실험합니다.

# torch.triu(m, diagonal=1): 행렬 m에서 "주대각선보다 위쪽"만 남기고 나머지는 0으로 만듭니다.
# diagonal=1이므로 주대각선(자기 자신 위치)은 포함하지 않습니다 -- 자기 자신은 봐도 되니까요.
future_mask = torch.triu(torch.ones(T_demo, T_demo), diagonal=1).bool()
print("future_mask (True = 미래라서 가려야 하는 위치):")
print(future_mask)
# 0행(0번째 토큰)은 자기 자신(0번)만 보고, 1~4번(미래)은 모두 True(가림)
# 4행(마지막 토큰)은 0~4번 전부를 볼 수 있으므로 전부 False

# 실제로는 Q, K를 이용해 계산되지만, 여기서는 마스킹 효과만 보기 위해
# "가짜" attention score를 무작위 숫자로 만듭니다.
scores_demo = torch.randn(T_demo, T_demo)
print("\n마스킹 전 attention score (무작위 예시 값):")
print(scores_demo)

# masked_fill: mask가 True인 위치를 지정한 값(-inf)으로 채웁니다.
scores_masked = scores_demo.masked_fill(future_mask, float('-inf'))
print("\n마스킹 후 attention score (미래 위치가 -inf로 바뀜):")
print(scores_masked)

# softmax를 적용하면 -inf였던 자리는 정확히 0이 됩니다.
probs_demo = F.softmax(scores_masked, dim=-1)
print("\nsoftmax 결과 (각 행이 하나의 확률분포입니다):")
print(probs_demo)

print("\n각 행의 합 (전부 1.0이어야 정상입니다 -- 확률의 정의):")
print(probs_demo.sum(dim=-1))

print("\n0번째 행에서 미래 위치(1~4번)의 확률이 정말 0인지 확인:")
print(probs_demo[0, 1:])

future_mask (True = 미래라서 가려야 하는 위치):
tensor([[False,  True,  True,  True,  True],
        [False, False,  True,  True,  True],
        [False, False, False,  True,  True],
        [False, False, False, False,  True],
        [False, False, False, False, False]])

마스킹 전 attention score (무작위 예시 값):
tensor([[ 0.3943,  0.1637,  1.0746,  0.8366,  0.0239],
        [-1.5204,  0.5294, -0.3908, -0.0877,  0.6329],
        [-1.0489, -0.5225,  0.7255, -0.9133,  1.9309],
        [-0.7463, -0.0111, -0.8344, -0.4922,  0.6580],
        [ 1.3285, -0.8279,  0.2825, -0.8119, -0.6384]])

마스킹 후 attention score (미래 위치가 -inf로 바뀜):
tensor([[ 0.3943,    -inf,    -inf,    -inf,    -inf],
        [-1.5204,  0.5294,    -inf,    -inf,    -inf],
        [-1.0489, -0.5225,  0.7255,    -inf,    -inf],
        [-0.7463, -0.0111, -0.8344, -0.4922,    -inf],
        [ 1.3285, -0.8279,  0.2825, -0.8119, -0.6384]])

softmax 결과 (각 행이 하나의 확률분포입니다):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1141, 0.8859, 0

## 5. Multi-Head Self-Attention 직접 구현하기

이제 위에서 배운 개념(임베딩, causal mask)을 합쳐서 **GPT의 핵심 부품인 Self-Attention**을 직접 만들어봅니다. 이 섹션이 이 노트북에서 가장 중요한 부분입니다.

### Query, Key, Value란 무엇인가?

Self-Attention을 처음 배울 때 가장 헷갈리는 부분이 Q(Query), K(Key), V(Value)라는 세 가지 벡터입니다. 도서관에서 책을 찾는 상황에 비유해봅시다.

- **Query(질문)**: "내가 지금 찾고 있는 것" — 예: "인공지능 역사에 관한 책을 찾고 싶다"
- **Key(색인/태그)**: "각 책에 붙어있는 색인 정보" — 예: 각 책 표지에 붙은 주제 태그
- **Value(실제 내용)**: "그 책의 실제 내용" — 태그가 내 질문과 잘 맞는 책일수록, 그 책의 내용을 더 많이 참고합니다.

즉, 내 Query와 각 위치의 Key를 비교해서 "얼마나 관련있는지(점수)"를 계산하고, 그 점수를 가중치 삼아 각 위치의 Value를 섞어서 가져오는 것이 attention입니다. 문장 속 모든 토큰이 각자 Query, Key, Value를 가지고 있고, "모든 토큰이 서로에게 동시에 이 질문을 하는 것"이 Self-Attention입니다.

### 왜 헤드(head)를 여러 개로 나눌까?

`num_heads`개의 헤드로 나눈다는 것은, `d_model` 차원 전체로 한 번에 "관련도"를 계산하는 대신, 차원을 `num_heads`개의 작은 조각(`d_head = d_model / num_heads`)으로 나눠서 **여러 개의 독립적인 "시선"으로 병렬 관찰**하는 것입니다. 예를 들어 한 헤드는 문법적 관계(주어-동사)에 집중하고, 다른 헤드는 의미적 유사성에 집중하는 식으로, 서로 다른 종류의 관계를 동시에 포착할 수 있게 해주는 장치입니다. (그래서 코드에서 `d_model`이 `num_heads`로 나누어 떨어져야 합니다 — 예: 16을 2개 헤드로 나누면 헤드 하나당 8차원.)

### 계산 순서 요약

1. 입력 `x`에 선형 변환(`nn.Linear`)을 적용해 Q, K, V를 만듭니다.
2. Q, K, V를 각각 `num_heads`개로 쪼갭니다.
3. 각 헤드마다 `Q @ K^T`로 "얼마나 관련있는지" 점수를 계산하고, `sqrt(d_head)`로 나눠 값이 너무 커지지 않게 조정합니다. (이 나눗셈이 없으면 차원이 커질수록 score의 분산이 커져서 softmax가 한쪽으로 너무 쏠리게 됩니다.)
4. 앞서 4번 섹션에서 실험했던 것과 **똑같은 방식**으로 causal mask를 적용해 미래 위치를 `-inf`로 가립니다.
5. `softmax`로 점수를 확률로 바꿉니다.
6. 그 확률로 V를 가중합(weighted sum)해서, "각 위치가 다른 위치의 정보를 얼마나 섞어서 가져올지" 계산합니다.
7. 나뉘어 있던 헤드를 다시 하나로 합치고, 마지막 선형 변환을 통과시켜 출력합니다.

> 참고: PyTorch에는 이미 완성된 `nn.MultiheadAttention`이 있어서 실무에서는 보통 이를 그대로 쓰거나, 더 빠른 `F.scaled_dot_product_attention`을 사용합니다. 하지만 이 노트북에서는 **내부에서 정확히 무슨 일이 일어나는지 보기 위해** 아래처럼 직접 구현합니다. 노트북 10번 섹션에서 우리가 만든 구현과 `nn.MultiheadAttention`을 나란히 비교해봅니다.

In [5]:
class CausalSelfAttention(nn.Module):
    """
    causal mask가 적용된 Multi-Head Self-Attention을 처음부터 직접 구현한 클래스.

    nn.MultiheadAttention을 쓰면 내부 연산이 함수 하나에 감춰지기 때문에,
    이 튜토리얼에서는 Q, K, V를 직접 만들고, attention score를 직접 계산하고,
    causal mask를 직접 적용하는 과정 전부를 forward() 안에서 눈으로 볼 수 있게 만듭니다.
    """

    def __init__(self, d_model, num_heads, max_seq_len, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model은 num_heads로 나누어 떨어져야 합니다."

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads  # 헤드 하나가 담당하는 차원 수

        # Q, K, V를 만드는 선형 변환을 하나로 합쳐서 한 번에 계산합니다.
        # (Linear를 3번 따로 호출하는 것과 결과는 같지만, 한 번의 행렬곱으로 처리해 더 효율적입니다.)
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)

        # 여러 헤드의 결과를 다시 하나로 합친 뒤 통과시키는 출력 투영층
        self.out_proj = nn.Linear(d_model, d_model)

        self.attn_dropout = nn.Dropout(dropout)   # attention 확률에 적용하는 dropout
        self.resid_dropout = nn.Dropout(dropout)  # 최종 출력에 적용하는 dropout

        # causal mask를 미리 만들어 등록해둡니다.
        # register_buffer로 등록하면 model.to(device) 같은 이동은 함께 되지만,
        # 학습 대상 파라미터로는 취급되지 않습니다 (마스크는 학습되는 값이 아니니까요).
        causal_mask = torch.triu(torch.ones(max_seq_len, max_seq_len), diagonal=1).bool()
        self.register_buffer("causal_mask", causal_mask)

    def forward(self, x, verbose=False):
        # x: [B, T, d_model]
        B, T, _ = x.shape

        # 1) Q, K, V를 한 번에 계산: [B, T, d_model] -> [B, T, 3*d_model]
        qkv = self.qkv_proj(x)
        # 마지막 차원을 3등분해서 Q, K, V로 나눕니다. 각각 [B, T, d_model]
        q, k, v = qkv.chunk(3, dim=-1)
        if verbose:
            print(f"  [1] Q/K/V 분리 직후 shape: {tuple(q.shape)}  (B={B}, T={T}, d_model={self.d_model})")

        # 2) num_heads개의 헤드로 나눕니다.
        # [B, T, d_model] -> [B, T, num_heads, d_head] -> [B, num_heads, T, d_head]
        # transpose(1, 2)로 "헤드 축"을 앞으로 옮겨서, 헤드별로 독립적인 행렬곱을 할 수 있게 합니다.
        q = q.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.d_head).transpose(1, 2)
        if verbose:
            print(f"  [2] 헤드로 분리 후 shape: {tuple(q.shape)}  (B, num_heads={self.num_heads}, T, d_head={self.d_head})")

        # 3) attention score 계산: Q와 K를 내적해서 "얼마나 관련있는지" 점수를 만듭니다.
        # [B, num_heads, T, d_head] @ [B, num_heads, d_head, T] -> [B, num_heads, T, T]
        # sqrt(d_head)로 나누는 이유: 차원이 커질수록 내적 값의 분산이 커져 softmax가
        # 지나치게 한쪽으로 쏠리는 것을 막기 위한 스케일링입니다.
        attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_head)
        if verbose:
            print(f"  [3] attention score shape: {tuple(attn_scores.shape)}  (T x T 행렬이 배치.헤드마다 하나씩)")

        # 4) causal mask 적용: 현재 시퀀스 길이(T)에 맞게 마스크를 잘라서 사용합니다.
        # (max_seq_len 크기로 미리 만들어둔 마스크 중 앞쪽 T x T 부분만 씁니다.)
        mask = self.causal_mask[:T, :T]
        attn_scores = attn_scores.masked_fill(mask, float('-inf'))

        # 5) softmax로 점수를 확률로 바꿉니다. (마지막 차원 = "어떤 위치를 볼지"에 대해 정규화)
        attn_probs = F.softmax(attn_scores, dim=-1)
        if verbose:
            # dropout을 적용하기 전에 확인해야 합이 정확히 1.0으로 나옵니다.
            row_sum = attn_probs[0, 0, -1].sum().item()
            print(f"  [4] softmax 직후 마지막 위치의 확률 합: {row_sum:.4f}  (1.0이어야 정상)")
        attn_probs = self.attn_dropout(attn_probs)

        # 6) 확률로 V를 가중합: 관련도가 높은 위치의 Value를 더 많이 가져옵니다.
        # [B, num_heads, T, T] @ [B, num_heads, T, d_head] -> [B, num_heads, T, d_head]
        attn_out = attn_probs @ v

        # 7) 나뉘어 있던 헤드를 다시 하나로 합칩니다.
        # [B, num_heads, T, d_head] -> [B, T, num_heads, d_head] -> [B, T, d_model]
        # transpose 이후 메모리가 연속적이지 않으므로 view 전에 contiguous()가 필요합니다.
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, T, self.d_model)
        if verbose:
            print(f"  [5] 헤드를 합친 후 shape: {tuple(attn_out.shape)}  (다시 B, T, d_model)")

        # 8) 출력 투영 + dropout
        out = self.resid_dropout(self.out_proj(attn_out))
        return out

In [6]:
attn_module = CausalSelfAttention(d_model, num_heads, max_seq_len)

print("입력 x shape:", x.shape)
print("\nCausalSelfAttention 내부 shape 추적:")
attn_out = attn_module(x, verbose=True)

print("\n최종 출력 attn_out shape:", attn_out.shape)
print("입력과 출력 shape이 같은가?", attn_out.shape == x.shape)

입력 x shape: torch.Size([2, 6, 16])

CausalSelfAttention 내부 shape 추적:
  [1] Q/K/V 분리 직후 shape: (2, 6, 16)  (B=2, T=6, d_model=16)
  [2] 헤드로 분리 후 shape: (2, 2, 6, 8)  (B, num_heads=2, T, d_head=8)
  [3] attention score shape: (2, 2, 6, 6)  (T x T 행렬이 배치.헤드마다 하나씩)
  [4] softmax 직후 마지막 위치의 확률 합: 1.0000  (1.0이어야 정상)
  [5] 헤드를 합친 후 shape: (2, 6, 16)  (다시 B, T, d_model)

최종 출력 attn_out shape: torch.Size([2, 6, 16])
입력과 출력 shape이 같은가? True


## 6. Feed-Forward Network(FFN)

Attention이 "다른 토큰들의 정보를 섞어오는" 역할을 했다면, Feed-Forward Network(FFN)는 각 위치(토큰)마다 **독립적으로** 한 번 더 비선형 변환을 가하는 역할을 합니다. (attention과 달리 FFN은 토큰들 사이의 정보 교환이 전혀 없습니다 — 오직 자기 자신의 벡터만 변환합니다.)

구조는 단순합니다: **차원을 확 늘렸다가(`d_model` → `d_ff`), 다시 줄입니다(`d_ff` → `d_model`)**. 보통 `d_ff`는 `d_model`의 4배 정도로 설정합니다 (GPT-2 Small: 768 → 3072 → 768). 이렇게 중간에 차원을 넓히는 이유는, 더 넓은 공간에서 다양한 특징을 표현하도록 모델에 여유를 주기 위해서입니다.

중간에 사용하는 `GELU`는 `ReLU`(음수는 0, 양수는 그대로)와 비슷하지만 경계 부분이 부드럽게 이어지는 활성화 함수입니다. GPT 계열 모델에서 표준적으로 사용됩니다.

```
x  ([B, T, d_model])
  │
  ▼
Linear(d_model -> d_ff)     # 차원을 넓힘
  │
  ▼
GELU                        # 비선형성 추가
  │
  ▼
Dropout
  │
  ▼
Linear(d_ff -> d_model)     # 다시 원래 차원으로
  │
  ▼
Dropout
  │
  ▼
출력 ([B, T, d_model])       # 입력과 shape이 동일 -- 그래야 residual 연결이 가능합니다
```

FFN은 다음 섹션에서 `GPTBlock`을 조립할 때 `nn.Sequential`로 간단히 정의하므로, 별도의 클래스 없이 바로 사용합니다.

## 7. LayerNorm과 잔차 연결(Residual Connection)

블록을 조립하기 전에 마지막으로 두 가지 장치를 알아둬야 합니다.

### LayerNorm (Layer Normalization)

`LayerNorm`은 **각 토큰 벡터를** (배치나 다른 토큰과는 무관하게, `d_model` 차원 안에서) 평균 0, 분산 1이 되도록 정규화한 뒤, 학습 가능한 스케일/이동(scale/shift)을 적용하는 연산입니다. (이미지 모델에서 흔한 BatchNorm은 "배치 전체"를 기준으로 정규화하지만, LayerNorm은 "토큰 하나, 그 안의 feature들"을 기준으로 정규화한다는 점이 다릅니다. 문장마다 길이가 달라질 수 있는 언어 모델에는 LayerNorm이 더 적합합니다.)

값의 스케일이 층을 거치면서 너무 커지거나 작아지는 것을 막아, 학습을 안정적으로 만들어주는 역할을 합니다.

### 잔차 연결 (Residual Connection)

`x = x + sublayer(x)`처럼, 어떤 연산의 결과를 원래 입력에 **더해서** 다음 층으로 넘기는 방식입니다. GPTBlock 안에서 attention과 FFN 모두 이 방식을 사용합니다. 이렇게 하는 이유는 두 가지입니다.

1. **그래디언트가 잘 흐릅니다.** 역전파 시 `+` 연산은 그래디언트를 그대로 양쪽에 전달하기 때문에, 층이 아무리 깊어져도(GPT-2는 12층, 큰 모델은 수십~수백 층) 학습 신호가 사라지지 않고 잘 전달됩니다.
2. **원래 정보가 보존됩니다.** sublayer(attention이나 FFN)가 설령 별 도움이 안 되는 값을 내놓아도, 원래 입력 정보(`x`)는 그대로 다음 층에 전달되므로 정보 손실이 적습니다.

### Pre-LN 구조

LayerNorm을 sublayer(attention, FFN)에 **들어가기 전에** 적용하는 방식을 Pre-LN이라고 부르고, GPT-2 이후 표준적으로 쓰입니다. (초기 Transformer 논문은 sublayer를 통과한 "후에" LayerNorm을 적용하는 Post-LN 방식이었는데, 층이 깊어질수록 학습이 불안정해지는 문제가 있어 Pre-LN이 더 널리 쓰이게 되었습니다.)

아래는 이번 섹션에서 만들 `GPTBlock` 하나의 내부 구조입니다. 이 그림의 각 화살표가 코드 한 줄씩과 대응됩니다.

```
x ──┬─────────────────────────────────┐
    │                                  │
    ▼                                  │
 LayerNorm (ln1)                       │
    │                                  │
    ▼                                  │
 Masked Self-Attention (5번 섹션)        │
    │                                  │
    ▼                                  │
   (+) ◄───────────────────────────────  잔차 연결: 원래 x를 더함
    │
    │   (여기서부터 x가 갱신된 상태로 아래 반복)
    ├─────────────────────────────────┐
    │                                  │
    ▼                                  │
 LayerNorm (ln2)                       │
    │                                  │
    ▼                                  │
 Feed-Forward Network (6번 섹션)         │
    │                                  │
    ▼                                  │
   (+) ◄───────────────────────────────  잔차 연결
    │
    ▼
 출력 (다음 GPTBlock으로, shape은 입력과 동일)
```

In [7]:
class GPTBlock(nn.Module):
    """
    GPT의 기본 블록 하나.
    구조: LayerNorm -> Self-Attention -> 잔차연결 -> LayerNorm -> FFN -> 잔차연결
    이 블록을 여러 겹 쌓은 것이 GPT 모델의 몸통입니다.
    """

    def __init__(self, d_model, num_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x, verbose=False):
        # -- Self-Attention 서브층 (Pre-LN) --
        # LayerNorm을 먼저 적용한 뒤 attention에 입력하고,
        # 그 결과를 "원래 x"에 더합니다 (잔차 연결).
        attn_out = self.attn(self.ln1(x), verbose=verbose)
        x = x + attn_out

        # -- Feed-Forward 서브층 (Pre-LN) --
        ffn_out = self.ffn(self.ln2(x))
        x = x + ffn_out

        if verbose:
            print(f"  [GPTBlock] 블록 통과 후 출력 shape: {tuple(x.shape)}  (입력과 동일해야 여러 블록을 쌓을 수 있음)")
        return x

In [8]:
block = GPTBlock(d_model, num_heads, d_ff, max_seq_len)

print("블록 입력 x shape:", x.shape)
block_out = block(x, verbose=True)
print("블록 출력 shape:", block_out.shape)
print("입력과 출력 shape이 같은가?", block_out.shape == x.shape)

블록 입력 x shape: torch.Size([2, 6, 16])
  [1] Q/K/V 분리 직후 shape: (2, 6, 16)  (B=2, T=6, d_model=16)
  [2] 헤드로 분리 후 shape: (2, 2, 6, 8)  (B, num_heads=2, T, d_head=8)
  [3] attention score shape: (2, 2, 6, 6)  (T x T 행렬이 배치.헤드마다 하나씩)
  [4] softmax 직후 마지막 위치의 확률 합: 1.0000  (1.0이어야 정상)
  [5] 헤드를 합친 후 shape: (2, 6, 16)  (다시 B, T, d_model)
  [GPTBlock] 블록 통과 후 출력 shape: (2, 6, 16)  (입력과 동일해야 여러 블록을 쌓을 수 있음)
블록 출력 shape: torch.Size([2, 6, 16])
입력과 출력 shape이 같은가? True


## 8. 전체 GPT 모델 조립하기

이제 지금까지 만든 부품들을 모두 조립합니다. 1번 섹션에서 본 큰 그림 그대로, 임베딩 → GPTBlock × num_layers → 최종 LayerNorm → Linear head 순서로 이어붙입니다.

`head`가 만들어내는 `logits`(shape: `[B, T, vocab_size]`)는 각 위치에서 "다음 토큰으로 각 단어가 나올 점수"입니다. 여기에 softmax를 적용하면 실제 확률 분포가 됩니다. (`head`에서는 아직 softmax를 적용하지 않고 raw score(logits) 상태로 반환하는데, 이렇게 하는 이유는 학습 시 사용하는 `F.cross_entropy` 손실 함수가 내부적으로 softmax까지 함께 계산해주기 때문입니다. 이 노트북에서는 학습을 다루지 않지만, 다음 실습을 위해 관례를 맞춰둡니다.)

### Weight Tying (가중치 공유)

아래 코드에 있는 다음 한 줄을 눈여겨봐 주세요.

```python
self.head.weight = self.tok_emb.weight
```

`tok_emb`(토큰 → 벡터로 변환하는 표)와 `head`(벡터 → 각 토큰의 점수로 변환하는 표)는 사실 **정반대 방향의 변환**이지만, 둘 다 `(vocab_size, d_model)` 크기의 행렬입니다. GPT-2는 이 두 행렬을 아예 같은 파라미터로 묶어서 씁니다 — 이것을 **weight tying(가중치 공유)** 이라고 부릅니다.

이렇게 하면 두 가지 이점이 있습니다.

1. **파라미터 수가 크게 줄어듭니다.** `vocab_size × d_model`만큼의 파라미터를 통째로 절약합니다. (GPT-2 Small 기준으로 약 3,860만 개 — 전체 파라미터의 약 30%에 해당하는 큰 절약입니다! 11번 섹션에서 직접 숫자로 확인해봅니다.)
2. **의미적으로도 자연스럽습니다.** "이 벡터가 특정 토큰과 얼마나 비슷한가"를 계산하는 것은, 그 토큰을 임베딩할 때 쓰는 벡터와 방향이 비슷할수록 자연스러우므로 두 표를 공유하는 것이 직관적으로도 말이 됩니다.

In [9]:
class GPT(nn.Module):
    def __init__(self, vocab_size, d_model=768, num_heads=12,
                 d_ff=3072, num_layers=12, max_seq_len=1024, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len

        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_seq_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)

        # GPTBlock을 num_layers개만큼 쌓습니다.
        self.blocks = nn.ModuleList([
            GPTBlock(d_model, num_heads, d_ff, max_seq_len, dropout)
            for _ in range(num_layers)
        ])

        self.ln_f = nn.LayerNorm(d_model)                       # 마지막 정규화
        self.head = nn.Linear(d_model, vocab_size, bias=False)  # d_model -> vocab_size

        # Weight tying: 입력 임베딩과 출력 head의 가중치를 공유합니다.
        self.head.weight = self.tok_emb.weight

    def forward(self, idx, verbose=False):
        # idx: [B, T] 정수 텐서 (토큰 id들)
        B, T = idx.shape
        assert T <= self.max_seq_len, (
            f"시퀀스 길이 {T}가 max_seq_len({self.max_seq_len})을 초과했습니다."
        )

        pos = torch.arange(T, device=idx.device)  # [0, 1, ..., T-1]

        tok_e = self.tok_emb(idx)   # [B, T, d_model]
        pos_e = self.pos_emb(pos)   # [T, d_model] -> broadcasting되어 더해짐
        x = self.emb_dropout(tok_e + pos_e)
        if verbose:
            print(f"  [GPT] 임베딩 합산 후 shape: {tuple(x.shape)}")

        # 블록을 순서대로 통과합니다.
        # (첫 블록만 자세한 로그를 남겨서 출력이 너무 길어지지 않게 합니다.)
        for i, block in enumerate(self.blocks):
            x = block(x, verbose=(verbose and i == 0))

        x = self.ln_f(x)
        logits = self.head(x)  # [B, T, vocab_size]
        if verbose:
            print(f"  [GPT] 최종 logits shape: {tuple(logits.shape)}  (B, T, vocab_size={self.head.out_features})")

        return logits

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        """
        idx: [B, T] 시작 토큰들 (프롬프트)
        max_new_tokens: 새로 생성할 토큰 개수
        temperature: 확률 분포를 얼마나 뾰족하게/평평하게 만들지 (다음 섹션에서 자세히 설명)
        top_k: 지정하면, 확률이 높은 top_k개의 토큰 중에서만 샘플링합니다.
        """
        self.eval()  # dropout 등을 비활성화합니다 (생성 시에는 원치 않는 무작위성이므로)
        for _ in range(max_new_tokens):
            # 시퀀스가 max_seq_len보다 길어지면 최근 max_seq_len개만 사용합니다.
            idx_cond = idx[:, -self.max_seq_len:]

            # 마지막 위치의 logits만 사용합니다 (그 다음에 올 토큰을 예측하는 부분이므로).
            logits = self(idx_cond)[:, -1, :] / temperature

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                # top_k 밖의 값들은 -inf로 만들어 샘플링 후보에서 제외합니다.
                logits[logits < v[:, [-1]]] = float('-inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # 확률에 따라 토큰 하나 샘플링
            idx = torch.cat([idx, next_token], dim=1)  # 생성된 토큰을 시퀀스 뒤에 이어 붙임
        return idx

In [10]:
# 앞서 정의한 작은 하이퍼파라미터로, 2개 블록짜리 미니 GPT를 만들어봅니다.
toy_model = GPT(
    vocab_size=vocab_size,
    d_model=d_model,
    num_heads=num_heads,
    d_ff=d_ff,
    num_layers=2,           # 실제 GPT-2 Small은 12개지만, 데모용으로 2개만 사용합니다.
    max_seq_len=max_seq_len,
)

print("입력 token_ids shape:", token_ids.shape)
logits = toy_model(token_ids, verbose=True)

print("\n최종 logits shape:", logits.shape)
print("기대하는 shape: (batch_size, seq_len, vocab_size) =", (batch_size, seq_len, vocab_size))

total_params = sum(p.numel() for p in toy_model.parameters())
print(f"\n이 미니 GPT의 전체 파라미터 수: {total_params:,}")

입력 token_ids shape: torch.Size([2, 6])
  [GPT] 임베딩 합산 후 shape: (2, 6, 16)
  [1] Q/K/V 분리 직후 shape: (2, 6, 16)  (B=2, T=6, d_model=16)
  [2] 헤드로 분리 후 shape: (2, 2, 6, 8)  (B, num_heads=2, T, d_head=8)
  [3] attention score shape: (2, 2, 6, 6)  (T x T 행렬이 배치.헤드마다 하나씩)
  [4] softmax 직후 마지막 위치의 확률 합: 1.0000  (1.0이어야 정상)
  [5] 헤드를 합친 후 shape: (2, 6, 16)  (다시 B, T, d_model)
  [GPTBlock] 블록 통과 후 출력 shape: (2, 6, 16)  (입력과 동일해야 여러 블록을 쌓을 수 있음)
  [GPT] 최종 logits shape: (2, 6, 50)  (B, T, vocab_size=50)

최종 logits shape: torch.Size([2, 6, 50])
기대하는 shape: (batch_size, seq_len, vocab_size) = (2, 6, 50)

이 미니 GPT의 전체 파라미터 수: 7,904


## 9. 텍스트 생성하기 (Autoregressive Generation)

지금까지 만든 `forward()`는 입력 시퀀스 전체에 대해 "각 위치에서 다음 토큰의 점수(logits)"를 한 번에 계산합니다. 실제로 새로운 문장을 만들어내려면, 이 점수를 이용해 **한 토큰씩 순차적으로 생성**해야 합니다. 이 과정을 **autoregressive(자기회귀적) generation**이라고 부릅니다.

**`generate()`의 반복 과정:**

1. 지금까지의 시퀀스를 모델에 넣어 `logits`를 계산합니다.
2. 그중 **마지막 위치**의 logits만 사용합니다 (그 다음에 올 토큰을 예측하는 부분이기 때문입니다).
3. `softmax`로 확률 분포를 만들고, 그 확률에 따라 토큰 하나를 무작위로 뽑습니다 (`torch.multinomial`).
4. 뽑은 토큰을 시퀀스 뒤에 이어 붙입니다.
5. 원하는 길이가 될 때까지 1~4를 반복합니다.

### temperature(온도)란?

`logits / temperature`로 나누는 부분이 있습니다. temperature는 확률 분포를 얼마나 "뾰족하게" 또는 "평평하게" 만들지 조절하는 값입니다.

- `temperature < 1.0` (예: 0.5): 원래도 확률이 높았던 토큰이 더욱 강조되어, 모델이 **가장 그럴듯한 답**을 반복적으로 선택하는 경향이 강해집니다. (보수적이고 예측 가능한 출력)
- `temperature = 1.0`: 모델이 학습한 확률 분포를 그대로 사용합니다.
- `temperature > 1.0` (예: 1.5): 확률 분포가 평평해져서, 원래 확률이 낮았던 토큰도 뽑힐 기회가 늘어납니다. (다양하지만 이상한 출력이 나올 위험도 커짐)

### top_k란? (보너스)

`top_k`를 지정하면, 확률이 가장 높은 `k`개의 토큰만 후보로 남기고 나머지는 확률을 0으로 만든 뒤 샘플링합니다. temperature가 높아서 이상한 토큰이 뽑힐 위험이 있을 때, "그래도 말이 되는 후보들 중에서만 고르자"는 안전장치 역할을 합니다.

> 참고: 아래 데모는 **학습을 전혀 시키지 않은** 모델을 사용합니다. 따라서 생성되는 토큰 자체(숫자)에는 아무 의미가 없습니다. 이 섹션에서 확인해야 할 것은 "어떤 토큰이 나오는가"가 아니라, "매 반복마다 시퀀스 길이가 1씩 늘어나며 shape이 올바르게 변하는가"입니다.

In [11]:
# 시작 프롬프트로, token_ids의 앞 2개 토큰만 사용합니다.
prompt = token_ids[:, :2]
print("프롬프트 shape:", prompt.shape)
print("프롬프트:", prompt)

# temperature=1.0 (학습된 확률을 그대로 사용)
generated = toy_model.generate(prompt, max_new_tokens=5, temperature=1.0)
print("\n[temperature=1.0] 생성 결과 shape:", generated.shape)
print(generated)

# top_k=5 를 추가로 적용한 예시
generated_topk = toy_model.generate(prompt, max_new_tokens=5, temperature=0.8, top_k=5)
print("\n[temperature=0.8, top_k=5] 생성 결과 shape:", generated_topk.shape)
print(generated_topk)

# 생성 전 프롬프트 길이 2 + 새로 생성한 5개 = 총 7개 토큰이 되었는지 확인
print("\n생성 후 길이가 프롬프트 길이 + max_new_tokens와 같은가?",
      generated.shape[1] == prompt.shape[1] + 5)

프롬프트 shape: torch.Size([2, 2])
프롬프트: tensor([[42, 17],
        [20, 24]])

[temperature=1.0] 생성 결과 shape: torch.Size([2, 7])
tensor([[42, 17, 17, 17, 17, 17,  2],
        [20, 24, 24, 24, 24, 24, 24]])

[temperature=0.8, top_k=5] 생성 결과 shape: torch.Size([2, 7])
tensor([[42, 17, 17, 17, 17, 17, 17],
        [20, 24, 24, 24, 24, 24, 13]])

생성 후 길이가 프롬프트 길이 + max_new_tokens와 같은가? True


## 10. (참고) PyTorch 내장 `nn.MultiheadAttention`과 비교

지금까지 Q/K/V 계산부터 causal mask, softmax까지 전부 직접 구현해봤습니다. 실무에서 GPT 계열 모델을 만들 때는 보통 이렇게 매번 직접 구현하기보다, PyTorch가 제공하는 검증되고 최적화된 구현을 사용합니다.

같은 입력 `x`와 같은 causal mask를 `nn.MultiheadAttention`에 넣으면, 우리가 만든 `CausalSelfAttention`과 **개념적으로 동일한 연산**(Q/K/V 계산 → score → mask → softmax → V 가중합)을 수행합니다. 다만 내부적으로 더 최적화되어 있고, attention weight(각 위치가 다른 위치를 얼마나 참고했는지)를 결과로 함께 돌려줍니다.

> 실무 팁: 최신 PyTorch에는 `F.scaled_dot_product_attention`이라는 함수도 있는데, 하드웨어에 따라 FlashAttention 같은 더 빠른 커널을 자동으로 선택해줍니다. 개념은 이 노트북에서 만든 것과 동일하지만 속도가 훨씬 빠릅니다. 원리를 이해했다면, 실전 코드에서는 이런 최적화된 함수를 사용하는 것이 좋습니다.

In [12]:
# batch_first=True: 입력 shape을 [B, T, d_model] 순서로 다루겠다는 옵션 (우리 구현과 동일한 순서)
mha = nn.MultiheadAttention(d_model, num_heads, batch_first=True)

# nn.MultiheadAttention은 mask를 우리처럼 미리 등록해두지 않고, 매번 인자로 넘겨줍니다.
causal_mask_bool = torch.triu(torch.ones(seq_len, seq_len), diagonal=1).bool()

# self-attention이므로 query, key, value에 모두 같은 x를 넣습니다.
mha_out, mha_weights = mha(x, x, x, attn_mask=causal_mask_bool)

print("nn.MultiheadAttention 출력 shape:", mha_out.shape)
print("attention weight shape:", mha_weights.shape, " (각 위치가 다른 위치를 얼마나 참고했는지)")

print("\n우리가 만든 CausalSelfAttention 출력 shape:", attn_out.shape)
print("-> shape이 동일한 것을 확인할 수 있습니다 (가중치는 각자 무작위 초기화라 값 자체는 다릅니다).")

nn.MultiheadAttention 출력 shape: torch.Size([2, 6, 16])
attention weight shape: torch.Size([2, 6, 6])  (각 위치가 다른 위치를 얼마나 참고했는지)

우리가 만든 CausalSelfAttention 출력 shape: torch.Size([2, 6, 16])
-> shape이 동일한 것을 확인할 수 있습니다 (가중치는 각자 무작위 초기화라 값 자체는 다릅니다).


## 11. GPT-2 Small 규모로 파라미터 수 확인하기

마지막으로, 지금까지 만든 `GPT` 클래스를 **실제 GPT-2 Small과 동일한 하이퍼파라미터**로 만들어 파라미터 개수를 확인해봅니다. (학습은 시키지 않고, 모델 구조와 크기만 확인합니다.)

| 하이퍼파라미터 | 값 |
|---|---|
| vocab_size | 50,257 |
| d_model | 768 |
| num_heads | 12 |
| d_ff | 3,072 |
| num_layers | 12 |
| max_seq_len | 1,024 |

이 값을 그대로 넣으면 파라미터 개수가 약 1.24억(124M) 개가 나와야 합니다 — 이것이 바로 "GPT-2 Small"이라는 이름의 유래입니다 (더 큰 버전인 Medium/Large/XL은 층 수와 차원이 더 큽니다).

아래 코드에서는 전체 파라미터 수와 함께, 8번 섹션에서 배운 **weight tying으로 절약된 파라미터 수**도 함께 계산해서 실제로 얼마나 큰 절약인지 확인해봅니다.

In [13]:
# 실제 GPT-2 Small과 동일한 하이퍼파라미터
GPT2_SMALL_VOCAB_SIZE = 50257
GPT2_SMALL_D_MODEL = 768
GPT2_SMALL_NUM_HEADS = 12
GPT2_SMALL_D_FF = 3072
GPT2_SMALL_NUM_LAYERS = 12
GPT2_SMALL_MAX_SEQ_LEN = 1024

gpt2_small = GPT(
    vocab_size=GPT2_SMALL_VOCAB_SIZE,
    d_model=GPT2_SMALL_D_MODEL,
    num_heads=GPT2_SMALL_NUM_HEADS,
    d_ff=GPT2_SMALL_D_FF,
    num_layers=GPT2_SMALL_NUM_LAYERS,
    max_seq_len=GPT2_SMALL_MAX_SEQ_LEN,
)

total_params = sum(p.numel() for p in gpt2_small.parameters())
print(f"Parameters: {total_params:,}")  # ~124M 이 나오는지 확인합니다.

# weight tying으로 얼마나 절약되었는지 확인합니다.
tied_matrix_params = gpt2_small.tok_emb.weight.numel()
print(f"\ntok_emb / head가 공유하는 행렬의 파라미터 수: {tied_matrix_params:,}")
print(f"만약 weight tying을 하지 않았다면 총 파라미터 수: {total_params + tied_matrix_params:,}")
print(f"weight tying으로 절약한 비율: {tied_matrix_params / (total_params + tied_matrix_params) * 100:.1f}%")

Parameters: 124,439,808

tok_emb / head가 공유하는 행렬의 파라미터 수: 38,597,376
만약 weight tying을 하지 않았다면 총 파라미터 수: 163,037,184
weight tying으로 절약한 비율: 23.7%


## 12. 정리 및 다음 단계

이 노트북에서 직접 구현하고 확인한 내용을 정리하면 다음과 같습니다.

- **토큰 임베딩 + 위치 임베딩**: 토큰 id를 의미 있는 벡터로 바꾸고, 순서 정보를 더했습니다.
- **Causal Mask**: `-inf`로 미래 위치를 가리면, softmax를 거친 뒤 그 확률이 정확히 0이 되는 것을 확인했습니다.
- **Multi-Head Self-Attention**: Q/K/V를 직접 계산하고, 헤드로 나누고, score를 구하고, mask를 적용하고, softmax와 V의 가중합까지 한 줄씩 shape을 추적하며 구현했습니다.
- **Feed-Forward Network / LayerNorm / 잔차 연결**: 각 서브층이 어떤 역할을 하는지, 왜 입력과 출력 shape이 항상 동일하게 유지되어야 하는지 확인했습니다.
- **GPTBlock과 GPT 조립**: 블록을 쌓아 전체 모델을 만들고, weight tying으로 파라미터를 절약하는 방법을 확인했습니다.
- **텍스트 생성**: temperature와 top_k를 이용한 autoregressive 샘플링을 구현했습니다.
- 마지막으로 실제 **GPT-2 Small(약 1.24억 파라미터)** 규모로 확장해 구조가 그대로 맞는 것을 확인했습니다.

### 직접 실험해보기

아래와 같이 값을 바꿔가며 다시 실행해보면 이해에 큰 도움이 됩니다.

- `num_heads`를 2 → 4 또는 8로 바꾸면 `d_head`와 attention score의 shape이 어떻게 바뀌나요? (단, `d_model`로 나누어 떨어지는 값이어야 합니다.)
- `seq_len`을 6 → 10으로 늘리면 causal mask 행렬은 어떻게 바뀌나요? (4번 섹션의 `torch.triu(...)` 코드를 다시 실행해보세요.)
- `torch.manual_seed()`의 숫자를 바꾸면 초기화되는 값과 생성 결과가 어떻게 달라지나요?
- `temperature`를 0.1과 2.0으로 각각 설정하고 `generate()`를 여러 번 실행하면, 생성되는 토큰들이 얼마나 다양해지나요?
- `dropout` 값을 0으로 바꾸면 어떤 차이가 있을까요? (`model.eval()`과 `model.train()` 상태에서 각각 실행해서 비교해보세요.)

### 이 노트북에서 다루지 않은 것들

이 노트북은 **모델 구조(architecture)** 자체에 집중했습니다. 실제로 GPT를 "학습"시키려면 아래와 같은 요소들이 추가로 필요하며, 이는 이후 실습에서 다룰 주제입니다.

- 텍스트를 토큰 id로 바꿔주는 **토크나이저(tokenizer)** (예: BPE)
- 학습 데이터셋 준비와 배치 구성
- 손실 함수(`F.cross_entropy`)와 역전파, optimizer를 이용한 학습 루프
- 학습된 모델을 평가하고 실제 텍스트를 생성해보는 과정

수고하셨습니다! 여기까지 직접 실행하고 값을 바꿔보셨다면, GPT 내부에서 텐서가 어떤 모양으로 흘러가는지에 대한 감을 확실히 잡으셨을 것입니다.